# 🎯 [DAT10_M08] 7일간 통신 단절 후 대량 재고 트랜잭션 일괄 동기화 스트레스 테스트
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT10/dat10_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT10_M08 |
| **미션 주제** | 7일간 통신 단절 후 대량 재고 트랜잭션 일괄 동기화 스트레스 테스트 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
전시에는 전술 통신망이 단절된 채 예하 보급소가 입고와 불출을 각자 기록하고, 통신 복구 후 기록을 병합하는 과정에서 중복·누락·충돌·위변조가 빈번히 발생하며, 이를 적시에 통제하지 못하면 재고 왜곡과 방산 공급망 위기 식별이 지연됩니다. 본 프로젝트는 방산 공급망 회복력 연구(미국 방위산업기반 정책연구)의 문제의식을 교육용 가상 자료로 재구성해, 가상 보급소 5곳이 통신 두절 환경에서 오프라인으로 재고를 기록하고 동기화하는 SaaS를 개발하고, 동기화된 데이터에서 공급망 재난 징후(업체 공급 감소, 재고 소진 임박)를 식별하는 End-to-End 군수 소프트웨어 엔지니어링 과제입니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 7일간 통신 단절 후 대량 재고 트랜잭션 일괄 동기화 스트레스 테스트
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

7일간 통신 단절 후 대량 재고 트랜잭션 일괄 동기화 스트레스 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **7일간 통신이 끊겼다가 복구된 직후 보급소 5곳의 누적 트랜잭션이 한꺼번에 서버로 몰리는 상황을 기준 100%·200%·500%로 시험해 시스템 다운 0건·데이터 유실 0%와 기능 축소 동작을 확인한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **트랜잭션 2,100건을 기준으로 4,200건·10,500건까지 늘려 배치 50건 단위로 큐와 작업자 스레드 4개가 처리하게 하고, 같은 배치 중복 재전송(5%)·서명 후 수량 변조(1%)·작업자 오류(37배치마다) 주입하여 처리 지연 및 복원력을 시험하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `tests/dat10_stress_test.py 및 스트레스 테스트 분석 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 극단 상황에서의 안전 실패(Fail-safe) 및 기능 축소 작동 적절성

#### 미션 PASS 조건

- 정량 기준: 동시 트래픽 500% 증가 시 시스템 다운 0건, 데이터 유실률 0%


- 진행 기준: 대량 시뮬레이션 부하 주입 테스트 통과 및 병목 구간 프로파일링 완료


- 검증 방법: 부하 테스트 도구를 통한 처리량(Throughput) 및 메모리 누수 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""7일간 통신이 끊긴 보급소 5곳의 누적 트랜잭션을 복구 직후 서버로 한꺼번에 올리는 상황을 기준 100%·200%·500%로 늘려 큐 기반 작업자 스레드로 처리하고, 중복 재전송·서명 불일치·작업자 오류를 주입해 다운 0건·유실 0건·기능 축소(표본 서명 점검 후 사후 검증)·병목(목록 탐색 중복 확인을 집합으로 교체)·메모리 증가를 측정한다."""
import hashlib
import hmac
import queue
import threading
import time
import tracemalloc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 야전 보급소·업체·품목 자료 생성기 (실제 부대·업체·품목·재고 정보 없음)
# 가상 보급소 5곳이 품목 12종의 입고·불출·조정 트랜잭션을 오프라인으로 기록하고 나중에 동기화한다.
# 공급망 재난 징후(업체 납기 지연, 품목 재고 급감)는 동기화된 자료를 마이닝해 식별한다.
# ---------------------------------------------------------------------------
NODES = ["보급소A", "보급소B", "보급소C", "보급소D", "보급소E"]
SUPPLIERS = ["가상업체1", "가상업체2", "가상업체3", "가상업체4", "가상업체5", "가상업체6"]
CATS = ["일반보급", "정비부품", "의무소모"]
ITEMS = [{"item": f"I{i + 1:02d}", "cat": CATS[i % 3], "supplier": SUPPLIERS[i % 6], "lead_days": 3 + (i * 5) % 12, "use": 4 + (i * 7) % 11, "min_level": 30 + 10 * (i % 5)} for i in range(12)]
ITEM_IDS = [x["item"] for x in ITEMS]
CTXS = ["평시", "야간", "통신불량"]


def make_txns(seed=7, days=7, per_node_day=60, nodes=None):
    """노드별 오프라인 트랜잭션 목록. id는 노드-일련번호(멱등 키), t는 시각(시간 단위), ctx는 입력 상황."""
    rng = np.random.default_rng(seed)
    out = []
    for n in (nodes or NODES):
        seq = 0
        for k in range(days * per_node_day):
            seq += 1
            it = ITEMS[int(rng.integers(len(ITEMS)))]
            t = float(rng.uniform(0, 24 * days))
            ctx = "야간" if (t % 24 >= 20 or t % 24 < 5) else "평시"
            if rng.random() < 0.12:
                ctx = "통신불량"
            r = rng.random()
            if r < 0.22:
                op, q = "in", int(it["use"] * rng.uniform(4, 9))
            elif r < 0.95:
                op, q = "out", int(max(1, rng.normal(it["use"] * 1.5, 3)))
            else:
                op, q = "adjust", int(rng.integers(-5, 6))
            out.append({"id": f"{n}-{seq:05d}", "node": n, "item": it["item"], "op": op, "qty": q, "t": round(t, 2), "ctx": ctx, "supplier": it["supplier"]})
    return out


def stock_of(txns, base=500):
    """품목별 재고 = 기초 재고 + 입고 - 불출 + 조정 (id 중복은 한 번만 반영)."""
    st, seen = {i: base for i in ITEM_IDS}, set()
    for x in txns:
        if x["id"] in seen:
            continue
        seen.add(x["id"])
        st[x["item"]] += x["qty"] if x["op"] in ("in", "adjust") else -x["qty"]
    return st

KEYS = {n: hashlib.sha256(f"demo-key-{n}".encode()).digest() for n in NODES}            # [교육용가상자료] 데모 키
BATCH, QMAX, HIGH_WATER, WORKERS = 50, 12, 8, 4


def sig_of(x):
    return hmac.new(KEYS[x["node"]], f"{x['id']}|{x['item']}|{x['op']}|{x['qty']}|{x['t']}".encode(), "sha256").hexdigest()[:16]


class Server:
    def __init__(self, fast):
        self.fast, self.lock = fast, threading.Lock()
        self.ids = set() if fast else []                                # 최적화 전: 목록 선형 탐색으로 중복 확인
        self.stock = {i: 500 for i in ITEM_IDS}; self.quar, self.dups, self.unver, self.lite_batches, self.applied = [], 0, [], 0, 0

    def ingest(self, batch, lite):
        ok_rows, unv = [], set()
        for k, x in enumerate(batch):
            if lite and k % 10:                                         # 기능 축소: 표본 10%만 서명 점검, 나머지는 사후 검증 대상
                good = True; unv.add(x["id"])
            else:
                good = hmac.compare_digest(sig_of(x), x["sig"])
            if not good:
                with self.lock:
                    self.quar.append(x["id"])
                continue
            ok_rows.append(x)
        with self.lock:
            self.lite_batches += lite
            for x in ok_rows:
                dup = x["id"] in self.ids
                if dup:
                    self.dups += 1; continue
                self.ids.add(x["id"]) if self.fast else self.ids.append(x["id"])
                self.stock[x["item"]] += x["qty"] if x["op"] in ("in", "adjust") else -x["qty"]; self.applied += 1
                if x["id"] in unv:
                    self.unver.append(x)                                # 실제로 반영된 미검증 레코드만 사후 검증 대상

    def post_verify(self):
        """사후 검증: 표본 점검만 거친 레코드를 전수 점검하고, 위조로 확인되면 반영분을 되돌린다."""
        rolled = 0
        for x in self.unver:
            if not hmac.compare_digest(sig_of(x), x["sig"]):
                self.stock[x["item"]] -= x["qty"] if x["op"] in ("in", "adjust") else -x["qty"]; self.quar.append(x["id"]); self.applied -= 1; rolled += 1
        return rolled


def run(mult, fast, seed=3):
    rng = np.random.default_rng(seed)
    tx = make_txns(7, 7, per_node_day=60 * mult)
    for x in tx:
        x["sig"] = sig_of(x)
    bad = set(rng.choice(len(tx), int(len(tx) * 0.01), replace=False).tolist())
    for i in bad:
        tx[i]["qty"] += 13                                              # 서명 후 수량 변조(1%)
    valid = [x for i, x in enumerate(tx) if i not in bad]
    truth = stock_of(valid)
    srv, q, lat, depth, dead, crash = Server(fast), queue.Queue(maxsize=QMAX), [], [], [], [0]
    lk = threading.Lock(); cnt = {"b": 0}

    def worker():
        while True:
            item = q.get()
            if item is None:
                return
            batch, t_in, retry = item
            depth.append(q.qsize())
            try:
                with lk:
                    cnt["b"] += 1; inj = (cnt["b"] % 37 == 0) and not retry                  # 37번째 배치마다 작업자 오류 주입(재시도 1회)
                if inj:
                    raise RuntimeError("worker failure(injected)")
                srv.ingest(batch, lite=fast and q.qsize() >= HIGH_WATER)
            except RuntimeError:
                dead.append((batch, t_in))                                                # 데드레터 보관 후 재처리(유실 방지)
            except Exception:
                crash[0] += 1
            lat.append((time.perf_counter() - t_in) * 1000)

    ths = [threading.Thread(target=worker) for _ in range(WORKERS)]
    [t.start() for t in ths]
    t0 = time.perf_counter()
    batches = []
    for n in NODES:
        mine = [x for x in tx if x["node"] == n]
        batches += [mine[i:i + BATCH] for i in range(0, len(mine), BATCH)]
    order = rng.permutation(len(batches)); resend = set(rng.choice(len(batches), max(1, len(batches) // 20), replace=False).tolist())
    for i in order:
        q.put((batches[i], time.perf_counter(), False))                                  # 가득 차면 생산자가 기다린다(역압, 유실 없음)
        if i in resend:
            q.put((batches[i], time.perf_counter(), False))                              # 같은 배치 중복 재전송
    for _ in ths:
        q.put(None)
    [t.join() for t in ths]
    for b, t_in in dead:
        srv.ingest(b, lite=False)
    rolled = srv.post_verify() if fast else 0
    dt = time.perf_counter() - t0
    diff = sum(abs(srv.stock[i] - truth[i]) for i in ITEM_IDS)
    lost = len(valid) - srv.applied
    return {"n": len(tx), "dt": dt, "tps": len(tx) / dt, "p95": float(np.percentile(lat, 95)), "qmax": max(depth) if depth else 0, "lite": srv.lite_batches, "dead": len(dead),
            "dups": srv.dups, "quar": len(set(srv.quar)), "bad": len(bad), "lost": lost, "diff": diff, "crash": crash[0], "rolled": rolled}


print(f"[7일 단절 후 일괄 동기화] 기준 {5 * 7 * 60}건(5곳 x 7일 x 60건) -> 100%/200%/500%, 배치 {BATCH}건, 작업자 {WORKERS}, 큐 {QMAX}, 중복 재전송 5%, 위조 1%, 오류 주입 37배치마다")
rows = []
for mult in (1, 2, 5):
    r0, r1 = run(mult, False), run(mult, True); rows.append((mult, r0, r1))
    print(f"  {mult * 100:3d}% ({r1['n']}건): 최적화 전 {r0['dt']:.2f}초({r0['tps']:.0f}건/초, p95 {r0['p95']:.0f}ms) -> 후 {r1['dt']:.2f}초({r1['tps']:.0f}건/초, p95 {r1['p95']:.0f}ms), 큐 최대 {r1['qmax']}, 축소 배치 {r1['lite']}")
print("\n[500% 상세] 항목: 최적화 전 / 후")
a, b = rows[-1][1], rows[-1][2]
for k, lab in (("dups", "중복 거부"), ("quar", "위조 격리"), ("dead", "작업자 오류 재처리 배치"), ("rolled", "사후 검증 되돌림"), ("lost", "유실 건수"), ("diff", "재고 차이 합계"), ("crash", "미처리 예외(다운)")):
    print(f"  {lab:16s} {a[k]:6d} / {b[k]:6d}")
lost_all = sum(r[1]["lost"] + r[2]["lost"] for r in rows); diff_all = sum(r[1]["diff"] + r[2]["diff"] for r in rows); crash_all = sum(r[1]["crash"] + r[2]["crash"] for r in rows)
quar_ok = all(r[2]["quar"] == r[2]["bad"] for r in rows)
print(f"\n전체 6회 실행: 다운 {crash_all}건, 유실 {lost_all}건, 재고 차이 {diff_all}, 위조 전부 격리 {'예' if quar_ok else '아니오'} (목표 다운 0건·유실 0%: {'달성' if crash_all == 0 and lost_all == 0 and diff_all == 0 else '미달'})")
print(f"병목 최적화(500%): 중복 확인을 목록 탐색 -> 집합 조회로 교체해 {a['dt']:.2f}초 -> {b['dt']:.2f}초({a['dt'] / b['dt']:.1f}배), 처리량 {a['tps']:.0f} -> {b['tps']:.0f}건/초")

# ---- 단계별 병목 프로파일링(수동 타이머): 서명 점검 / 중복 확인(목록·집합) ----
tx5 = make_txns(7, 7, per_node_day=300)
for x in tx5:
    x["sig"] = sig_of(x)
t0 = time.perf_counter(); [hmac.compare_digest(sig_of(x), x["sig"]) for x in tx5]; t_sig = time.perf_counter() - t0
ids = [x["id"] for x in tx5]; t0 = time.perf_counter(); lst = []
for i in ids:
    _ = i in lst; lst.append(i)
t_lst = time.perf_counter() - t0; t0 = time.perf_counter(); st = set()
for i in ids:
    _ = i in st; st.add(i)
t_set = time.perf_counter() - t0
print(f"\n[병목 프로파일] {len(ids)}건 기준: 서명 점검 {t_sig * 1000:.0f}ms, 중복 확인 목록 {t_lst * 1000:.0f}ms / 집합 {t_set * 1000:.1f}ms -> 병목은 중복 확인이었고 서명 점검이 다음 병목")
print("[기능 축소 정책] 큐가 %d건 이상 쌓이면 서명을 표본 10%%만 점검하고 나머지는 사후 전수 검증한다. 위조가 확인되면 되돌리고 격리한다. 데이터는 버리지 않는다." % HIGH_WATER)

# ---- 메모리 누수 점검: 같은 부하를 3회 반복해 남는 메모리 ----
tracemalloc.start(); mem = []
for _ in range(3):
    run(1, True); mem.append(tracemalloc.get_traced_memory()[0] / 1e6)
tracemalloc.stop()
print(f"[메모리] 반복 실행 후 남은 메모리 {', '.join(f'{m:.1f}MB' for m in mem)} -> 증가 {mem[-1] - mem[0]:+.1f}MB ({'누수 징후 없음' if mem[-1] - mem[0] < 5 else '확인 필요'})")
print("※ 처리 시간은 실행 환경에 따라 달라진다. 실제 서버는 별도 작업 큐·영속 저장소·부하 시험 도구가 필요하다. 이 시험은 단일 프로세스 모의이다.")

plt.figure(figsize=(7, 3.6)); x = np.arange(3)
plt.bar(x - .2, [r[1]["dt"] for r in rows], .4, label="before (list scan)", color="tab:gray"); plt.bar(x + .2, [r[2]["dt"] for r in rows], .4, label="after (set + lite mode)", color="tab:blue")
plt.xticks(x, ["100%", "200%", "500%"]); plt.ylabel("seconds"); plt.title("Bulk sync time after 7-day outage"); plt.legend(); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[7일 단절 후 일괄 동기화] 기준 2100건(5곳 x 7일 x 60건) -> 100%/200%/500%, 배치 50건, 작업자 4, 큐 12, 중복 재전송 5%, 위조 1%, 오류 주입 37배치마다
  100% (2100건): 최적화 전 0.05초(45132건/초, p95 21ms) -> 후 0.01초(161281건/초, p95 2ms), 큐 최대 11, 축소 배치 15
  200% (4200건): 최적화 전 0.16초(26448건/초, p95 45ms) -> 후 0.03초(126250건/초, p95 6ms), 큐 최대 11, 축소 배치 27
  500% (10500건): 최적화 전 1.00초(10489건/초, p95 134ms) -> 후 0.06초(165580건/초, p95 3ms), 큐 최대 11, 축소 배치 68

[500% 상세] 항목: 최적화 전 / 후
  중복 거부               496 /    497
  위조 격리               105 /    105
  작업자 오류 재처리 배치         5 /      5
  사후 검증 되돌림             0 /     32
  유실 건수                 0 /      0
  재고 차이 합계              0 /      0
  미처리 예외(다운)            0 /      0

전체 6회 실행: 다운 0건, 유실 0건, 재고 차이 0, 위조 전부 격리 예 (목표 다운 0건·유실 0%: 달성)
병목 최적화(500%): 중복 확인을 목록 탐색 -> 집합 조회로 교체해 1.00초 -> 0.06초(15.8배), 처리량 10489 -> 165580건/초

[병목 프로파일] 10500건 기준: 서명 점검 41ms, 중복 확인 목록 765ms / 집합 2.1ms -> 병목은 중복 확인이었고 서명 점검이 다음 병목
[기능 축소 정책] 큐가 8건 이상 쌓이면 서명을 표본 10%만 점검하고 나머지는 사후 전수 검증한다. 위조가 확인되면 되돌리고 격리한다. 데이터는 버리지 않는다.
[메모리] 반복 실행 후 남은 메모리 0.0MB, 0.0MB, 0.0MB -> 증가 +0.0MB (누수 징후 없음)
※ 처리 시간은 실행 환경에 따라 달라진다. 실제 서버는 별도 작업 큐·영속 저장소·부하 시험 도구가 필요하다. 이 시험은 단일 프로세스 모의이다.
```
</details>
